In [1]:
%pip -qU langchain-teddynote

Note: you may need to restart the kernel to use updated packages.



Usage:   
  d:\hanwha_0902\hanwha_0902\ex0916\.venv\Scripts\python.exe -m pip <command> [options]

no such option: -U


In [1]:
%pip --version

pip 25.0.1 from d:\hanwha_0902\hanwha_0902\ex0916\.venv\Lib\site-packages\pip (python 3.12)

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -qU langchain-teddynote

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("test_0916")

llm = ChatOpenAI(temperature=0, model_name="gpt-4o")

LangSmith 추적을 시작합니다.
[프로젝트명]
test_0916


In [ ]:
#StructuredOutputParser가 없어서 다른 걸로 대체함.

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("test_0916")

llm = ChatOpenAI(temperature=0, model_name="gpt-4o")

class AnswerWithSource(BaseModel):
    answer: str = Field(description="사용자 질문에 대한 답변 (한국어)")
    source: str = Field(description="답변의 근거가 된 출처(문서명, 자료명 등)")
    url: str = Field(description="답변의 근거가 된 웹사이트 주소(URL)")

parser = PydanticOutputParser(pydantic_object=AnswerWithSource)

prompt = PromptTemplate.from_template(
    """당신은 정확한 정보를 제공하는 AI 어시스턴트입니다.
아래 질문에 한국어로 답변하고, 답변의 근거가 된 출처와 웹사이트 주소를 함께 알려주세요.

QUESTION: {question}

FORMAT:
{format}
"""
).partial(format=parser.get_format_instructions())

chain = prompt | llm | parser

result = chain.invoke({"question": "대한민국의 맛집은 어디인가요?"})

print("답변:", result.answer)
print("출처:", result.source)
print("URL:", result.url)

In [11]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("test_0916")

llm = ChatOpenAI(temperature=0, model_name="gpt-4o")

class AnswerWithSource(BaseModel):
    answer: str = Field(description="사용자 질문에 대한 답변 (한국어)")
    source: str = Field(description="답변의 근거가 된 출처(문서명, 자료명 등)")
    url: str = Field(description="답변의 근거가 된 웹사이트 주소(URL)")

parser = PydanticOutputParser(pydantic_object=AnswerWithSource)

prompt = PromptTemplate.from_template(
    """당신은 정확한 정보를 제공하는 AI 어시스턴트입니다.
아래 질문에 한국어로 답변하고, 답변의 근거가 된 출처와 웹사이트 주소를 함께 알려주세요.

QUESTION: {question}

FORMAT:
{format}
"""
).partial(format=parser.get_format_instructions())

chain = prompt | llm | parser

result = chain.invoke({"question": "대한민국의 맛집은 어디인가요?"})

print("답변:", result.answer)
print("출처:", result.source)
print("URL:", result.url)

LangSmith 추적을 시작합니다.
[프로젝트명]
test_0916
답변: 대한민국에는 다양한 맛집이 있습니다. 예를 들어, 서울의 '광장시장'은 전통적인 한국 음식을 맛볼 수 있는 곳으로 유명하며, 부산의 '자갈치 시장'은 신선한 해산물을 즐길 수 있는 곳으로 잘 알려져 있습니다.
출처: 한국관광공사
URL: https://korean.visitkorea.or.kr


In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test_0916")

model = ChatOpenAI(temperature=0.2, model_name="gpt-5")

class Topic(BaseModel):
    description: str= Field(description="주제에 대한 간결한 설명")
    hashtags: str= Field(description="해시태그 형식의 키워드(2개 이상)")

question = "오늘 용산역 주위에 엄마와 함께 같이 갈만한 맛집 추천해주세요."

parser = JsonOutputParser(pydantic_object=Topic)
print(parser.get_format_instructions())

prompt= ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 AI 어시스턴트입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format : {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain=prompt | model | parser

answer = chain.invoke({"question": question})

answer["description"]

LangSmith 추적을 시작합니다.
[프로젝트명]
test_0916
STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any 

'용산역이라면 아이파크몰 식당가가 가장 무난해요. 엄마랑 편히 먹기 좋은 한식 백반·설렁탕/곰탕, 조용한 일식(우동·초밥), 넓은 좌석의 파스타·스테이크 매장이 많고 엘리베이터·주차가 편합니다. 조금 더 차분한 식사를 원하면 용산 드래곤시티 호텔 레스토랑(뷔페·한식·양식)이 좌석 간격이 넓어 어르신 동행에 좋아요. 가볍게 걸어가면 신용산·삼각지 골목의 칼국수·만두, 생선구이 백반, 갈비탕 같은 담백한 노포들도 좋아요. 오늘은 11:30 이전이나 13:30 이후 방문, 가능하면 예약을 추천합니다.'

In [27]:
#7 Pandas 활용
import pprint
from typing import Any, Dict

import pandas as pd
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test_0916")

class PandasQuery(BaseModel) :
    column: str = Field(description="분석할 컬럼 이름")
    operation: str = Field(
        description="수행할 연산. 예: mean, count, sum, min, max, quantile"
    )
    filter_column: str | None = Field(
        default=None,
        description="필터링에 사용할 컬럼"
    )
    filter_value: int | float | str | None = Field(
        default=None,
        description="필터링할 값"
    )

model = ChatOpenAI(temperature=0.5, model_name="gpt-4o-mini")
structured_model = model.with_structured_output(PandasQuery)

def format_parser_output(parser_output: Dict[str, Any]) -> None:
    for key in parser_output.keys():
        parser_output[key] = parser_output[key].to_dict()
    return pprint.PrettyPrinter(width=40, compact=True).pprint(parser_output)

df = pd.read_csv("./titanic.csv")
df.head(10)


LangSmith 추적을 시작합니다.
[프로젝트명]
test_0916


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [ ]:
%pip install pandas

In [ ]:
parser = PandasDataFrameOutputParser(dataframe=df)

print(parser.get_format_instructions())

In [34]:
prompt = f"""
다음 DataFrame의 컬럼은 아래와 같습니다.

{df.columns.tolist()}

사용자의 질문을 분석해서 PandasQuery 형식으로 반환하세요.
"""
pprint.pprint(df.columns.tolist())

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']


In [ ]:
%pip install langchain_classic

In [38]:
df_query="Age column을 조회해 주세요."

prompt=PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{question}\n",
    input_variables=["question"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)
chain = prompt | model | parser
parser_output = chain.invoke({"question": df_query})
pprint.pprint(parser_output)

{'description': 'Age column refers to the data field that contains the ages of '
                'individuals in a dataset.',
 'hashtags': '#age,#data'}


In [42]:
df_query = "Retrieve the first row."
parser_output= chain.invoke({"question": df_query})
pprint.pprint(parser_output)
df["Age"].head().mean()

{'description': '이 주제는 주어진 데이터에 대한 간결한 설명입니다.', 'hashtags': '#예시 #데이터'}


np.float64(31.2)

In [46]:
df_query="Retrieve the average of the Ages from row 0 to 810."
parser_output=chain.invoke({"question":df_query})
print(parser_output)

{'description': 'The average age calculated from the specified rows is 35.2 years.', 'hashtags': '#DataAnalysis,#AgeStatistics'}
